In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyfastx
from collections import defaultdict
import matplotlib.patches as mpatches
from viz_sequence_v2 import plot_dashboard_grid
import re
import os
from utils import parse_vadr_cds_tbl, format_cds_coordinates
from scipy.spatial.distance import cdist, cosine
from sklearn.preprocessing import normalize

In [2]:
def is_at_rich(codon):
    # Define as A/T-rich if majority of bases are A or T
    return sum(1 for b in codon if b in 'AT') >= 2

In [3]:
def batch_cosine_dist(X_query, ref_vector, batch_size=2000):
    """
    Computes cosine distance between many query vectors and a single reference vector.
    Optimization: Uses dot product of normalized vectors instead of cdist.
    
    Args:
        X_query (np.ndarray): Shape (N, features)
        ref_vector (np.ndarray): Shape (features,) or (1, features)
        batch_size (int): Size of chunks (less critical here but good for consistency)
        
    Returns:
        np.ndarray: Array of cosine distances of shape (N,)
    """
    # 1. Ensure ref_vector is shape (1, features)
    ref_vector = ref_vector.reshape(1, -1)
    ref_norm = normalize(ref_vector, norm='l2', axis=1)
    X_query_norm = normalize(X_query, norm='l2', axis=1)
    
    similarities = X_query_norm.dot(ref_norm.T).flatten()
    
    similarities = np.clip(similarities, -1.0, 1.0)
    distances = 1.0 - similarities
    
    return distances

In [4]:
df_train_shap     = pd.read_csv('/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/data/section3_4_5/01_PB2/All_True_Label_Training_01_PB2_SHAP.csv').reset_index(drop=True)
df_train_metadata = pd.read_csv('/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/metadata/01_PB2/01_PB2_Train_Filtered.csv').reset_index(drop=True)

In [5]:
df_train_shap.shape

(78732, 66)

In [6]:
df_metadata_H7N9_China = pd.read_csv('H7N9_China_E627K.csv')

In [7]:
df_merge = pd.merge(df_train_shap, df_metadata_H7N9_China, left_on='Seq_ID', right_on='1_PB2_ID', how='left')

In [8]:
df_merge = df_merge[['Isolate_Name', 'NCBI_Accession', '1_PB2_ID', 'host',
       'serotype', 'country', 'mutations', 'Publication', 'GCA',
       'GCC', 'GCG', 'GCT', 'TGC', 'TGT', 'GAC', 'GAT', 'GAA', 'GAG', 'TTC',
       'TTT', 'GGA', 'GGC', 'GGG', 'GGT', 'CAC', 'CAT', 'ATA', 'ATC', 'ATT',
       'AAA', 'AAG', 'CTA', 'CTC', 'CTG', 'CTT', 'TTA', 'TTG', 'ATG', 'AAC',
       'AAT', 'CCA', 'CCC', 'CCG', 'CCT', 'CAA', 'CAG', 'AGA', 'AGG', 'CGA',
       'CGC', 'CGG', 'CGT', 'AGC', 'AGT', 'TCA', 'TCC', 'TCG', 'TCT', 'ACA',
       'ACC', 'ACG', 'ACT', 'GTA', 'GTC', 'GTG', 'GTT', 'TGG', 'TAC', 'TAT',
        ]]

In [9]:
df_merge.index

RangeIndex(start=0, stop=78732, step=1)

In [10]:
df_merge.shape

(78732, 69)

In [11]:
df_merge.head()

,Isolate_Name,NCBI_Accession,1_PB2_ID,host,serotype,country,mutations,Publication,GCA,GCC,...,ACC,ACG,ACT,GTA,GTC,GTG,GTT,TGG,TAC,TAT
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000787,0.006989,...,0.004000,0.000811,0.033669,0.023245,0.014094,0.008750,0.048468,-0.007495,0.010470,-0.002706
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.002728,0.024686,...,0.028739,0.034733,0.023645,0.026851,-0.015415,0.014771,0.019005,0.010986,-0.022229,-0.003556
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.032759,0.031747,...,0.004079,0.052372,0.030776,0.039504,0.002358,0.070922,0.002953,-0.002107,0.008281,-0.003621
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.022239,-0.056647,...,0.005503,-0.010759,0.038610,0.035118,0.011972,0.017404,0.056612,-0.007028,0.004832,0.006667
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.005377,0.017649,...,-0.000057,0.005515,0.033465,0.028741,0.016244,-0.000290,0.051216,-0.006244,0.045682,0.002975


In [12]:
pred = np.load('/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/results/pre2020_calibration/01_PB2_preds_cal_fold_0.npy')

In [13]:
df_merge[df_merge['mutations']=='E627K']

,Isolate_Name,NCBI_Accession,1_PB2_ID,host,serotype,country,mutations,Publication,GCA,GCC,...,ACC,ACG,ACT,GTA,GTC,GTG,GTT,TGG,TAC,TAT
16228,A/Shanghai/1/2013,NaN,EPI439488,Human,H7N9,China,E627K,https://doi.org/10.2807/1560-7917.ES2014.19.25...,-0.011377,-0.011186,...,-0.009245,-0.007565,0.000124,0.006194,-0.006126,0.011663,-0.007681,0.000766,0.002093,0.003856
16229,A/Shanghai/2/2013,NaN,EPI439495,Human,H7N9,China,E627K,https://doi.org/10.2807/1560-7917.ES2014.19.25...,0.010068,-0.001621,...,-0.002243,0.009869,-0.001260,0.005568,-0.004238,-0.041829,-0.014477,-0.012126,0.006261,0.000453
16230,A/Anhui/1/2013,NaN,EPI439504,Human,H7N9,China,E627K,https://doi.org/10.2807/1560-7917.ES2014.19.25...,-0.014926,-0.008999,...,-0.006684,0.007133,-0.000955,-0.002707,-0.006537,-0.021202,-0.000656,-0.003579,0.002041,0.008090
16313,A/Shanghai/3/2013,NaN,EPI447961,Human,H7N9,China,E627K,https://doi.org/10.2807/1560-7917.ES2014.19.25...,-0.006431,-0.004941,...,0.017302,0.017255,-0.004919,0.014807,0.007195,-0.008323,0.021744,-0.010089,0.008286,0.001995
16447,A/Jiangsu/04/2013,NaN,EPI447852,Human,H7N9,China,E627K,https://doi.org/10.2807/1560-7917.ES2014.19.25...,-0.006580,-0.040213,...,-0.020722,-0.008360,0.003377,0.001217,-0.007160,-0.032666,0.014886,-0.017019,-0.003293,-0.000682
16450,A/Shanghai/10/2013,NaN,EPI447810,Human,H7N9,China,E627K,https://doi.org/10.2807/1560-7917.ES2014.19.25...,-0.004560,-0.006228,...,0.014005,0.009798,-0.005751,0.011814,0.006249,-0.020437,0.027284,-0.013817,0.012308,-0.002283
16451,A/Shanghai/11/2013,NaN,EPI447802,Human,H7N9,China,E627K,https://doi.org/10.2807/1560-7917.ES2014.19.25...,0.007842,-0.016293,...,0.007082,-0.004058,0.004324,0.016052,0.002912,-0.026457,-0.008000,-0.008469,0.001266,-0.008198
16452,A/Shanghai/13/2013,NaN,EPI447785,Human,H7N9,China,E627K,https://doi.org/10.2807/1560-7917.ES2014.19.25...,0.015618,-0.005915,...,-0.015134,0.000239,-0.000454,0.011529,0.004763,-0.028230,-0.016001,-0.012157,0.006010,0.002859
16453,A/Shanghai/14/2013,NaN,EPI447778,Human,H7N9,China,E627K,https://doi.org/10.2807/1560-7917.ES2014.19.25...,-0.013998,-0.007250,...,0.000482,-0.004213,-0.004471,0.007100,0.004965,-0.000285,0.002633,-0.004659,0.000553,0.001318
16454,A/Shanghai/17/2013,NaN,EPI447770,Human,H7N9,China,E627K,https://doi.org/10.2807/1560-7917.ES2014.19.25...,-0.014581,-0.004870,...,0.006828,0.006203,0.009383,-0.020231,0.007646,-0.010614,-0.005152,-0.000518,-0.000036,0.021171


In [14]:
df_merge.columns

Index(['Isolate_Name', 'NCBI_Accession', '1_PB2_ID', 'host', 'serotype',
       'country', 'mutations', 'Publication', 'GCA', 'GCC', 'GCG', 'GCT',
       'TGC', 'TGT', 'GAC', 'GAT', 'GAA', 'GAG', 'TTC', 'TTT', 'GGA', 'GGC',
       'GGG', 'GGT', 'CAC', 'CAT', 'ATA', 'ATC', 'ATT', 'AAA', 'AAG', 'CTA',
       'CTC', 'CTG', 'CTT', 'TTA', 'TTG', 'ATG', 'AAC', 'AAT', 'CCA', 'CCC',
       'CCG', 'CCT', 'CAA', 'CAG', 'AGA', 'AGG', 'CGA', 'CGC', 'CGG', 'CGT',
       'AGC', 'AGT', 'TCA', 'TCC', 'TCG', 'TCT', 'ACA', 'ACC', 'ACG', 'ACT',
       'GTA', 'GTC', 'GTG', 'GTT', 'TGG', 'TAC', 'TAT'],
      dtype='str')

In [15]:
df_merge = df_merge.reset_index(drop=True)

In [16]:
pred.shape

(78732,)

In [17]:
stop_codons = {'TAA', 'TAG', 'TGA'}

In [18]:
h7n9_fa = pyfastx.Fasta('H7N9_China.fasta', build_index=True)

In [19]:
X_onehot    = np.load('/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/data/section3_4_5/01_PB2/X_train_onehot.npy')
shap_scores = np.load('/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/data/section3_4_5/01_PB2/01_PB2_Training_OOF.npy')

In [20]:
shap_scores.shape

(78732, 5, 2400, 3)

In [21]:
def onehot_to_dna(onehot):
    # 1. Define the mapping based on your encoding order
    # Common order: A, C, G, T, N (or gap)
    alphabet = np.array(['A', 'C', 'G', 'T', 'N'])
    mask = onehot.any(axis=0)
    # 2. Find the index of the '1' in each column
    # onehot shape: (5, seq_len)
    indices = np.argmax(onehot[:, mask], axis=0)
    
    # 3. Map indices to the alphabet and join into a string
    dna_seq = "".join(alphabet[indices])
    
    return dna_seq

In [22]:
cds_tbl_path = f'01_PB2_VADR.vadr.pass.tbl'
parsed_cds_tbl_data = parse_vadr_cds_tbl(cds_tbl_path)
cds_coords_dict = format_cds_coordinates(parsed_cds_tbl_data)

In [23]:

supplementary_data = []

# Constants for E627K (Codon 627)
TARGET_CODON_NUM = 627
# 1-based codon -> 0-based nucleotide index
# (627 - 1) * 3 = 1878
START_NUC_IDX = (TARGET_CODON_NUM - 1) * 3  

results = "/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results"
if not os.path.exists(results):
    os.makedirs(results)

print(f"Starting analysis...")
codon_list = np.load('codon_list.npy')
print (codon_list)
avi_medoid = np.load('01_PB2_avi_medoid_pre2020.npy')
dists = []
for index, row in df_merge.iterrows():
    seq_id = row['1_PB2_ID']
    # FILTER: Human + E627K
    if row['host'] == 'Human' and row['mutations'] == 'E627K':
        
        seq = h7n9_fa[seq_id].seq.upper()
        clean_seq = re.sub(r'[^ACGT]', 'N', seq.upper())
        ## just check the data to make sure it is a correct sequence
        assert clean_seq == onehot_to_dna(X_onehot[index]) 
        print ('Model prediction:', pred[index])

        cds_coords = cds_coords_dict[seq_id]['polymerase PB2']
        extracted_coords = cds_coords['coordinate']
        codon_start = cds_coords.get('codon_start', 0)
        print (extracted_coords, codon_start)
        
        # --- A. Calculate Sequence Scores ---
        # Raw SHAP (5, L) * One-Hot (5, L) = Contribution of Actual Bases (5, L)
        raw_shap = shap_scores[index, 0:5, :, 0] # codon_start alsway 1, coord is start 1 for this dataset
        raw_onehot = X_onehot[index, 0:5, :]
        seq_scores = raw_shap * raw_onehot
        seq_scores = seq_scores[:, :2280] 
        
        # --- B. Get E627K Specific Scores (Positions 1, 2, 3) ---
        # START_NUC_IDX = 1878
        score_pos1 = np.sum(seq_scores[:, START_NUC_IDX])     # 1878 (A)
        score_pos2 = np.sum(seq_scores[:, START_NUC_IDX + 1]) # 1879 (A)
        score_pos3 = np.sum(seq_scores[:, START_NUC_IDX + 2]) # 1880 (G)
        
        total_627_score = score_pos1 + score_pos2 + score_pos3
        actual_codon_seq = seq[START_NUC_IDX : START_NUC_IDX + 3] # Should be AAG

        # --- C. Get Top 10 Positive Nucleotides Globally ---
        # Sum axis 0 -> 1D array of scores per position
        per_nuc_scores = np.sum(seq_scores, axis=0) 
        
        positive_sites = []
        for i, score in enumerate(per_nuc_scores):
            if score > 0:
                # Store: (Position 1-based, Score, Nucleotide)
                positive_sites.append((i + 1, score, seq[i]))
        
        # Sort descending
        positive_sites.sort(key=lambda x: x[1], reverse=True)
        top_20 = positive_sites[:20]
        
        # Format string: "Pos 1880(G): 0.52; Pos 120(A): 0.45"
        top_20_str = "; ".join([f"Pos {p}({n}): {s:.4f}" for p, s, n in top_20])


        # --- E. Plotting ---
        # 1. Codon scores for scatter
        codon_shap_by_pos = defaultdict(float)
        
        for pos, codon_idx in enumerate(range(0, len(seq), 3)):
            codon = seq[codon_idx:codon_idx+3]
            if codon =='AAG' or codon=='AAA':
                codon_shap_score = np.sum(seq_scores[:, codon_idx:codon_idx+3])
                codon_shap_by_pos[pos+1] = codon_shap_score

        # 2. Logo Data: Transpose (4, L) -> (L, 4)
        seq_logo_data = np.transpose(seq_scores[0:4, :]) # get rid of fifth element for plot
        
        codon_bar_data = row[codon_list].to_dict()

        #######################Check############################
        codon_score_check = {}
        for pos, codon_idx in enumerate(range(0, 2280, 3)):
            codon = clean_seq[codon_idx:codon_idx+3]
            if 'N' not in codon and codon not in stop_codons:
                codon_shap_score_check = np.sum(seq_scores[:, codon_idx:codon_idx+3])
                codon_score_check[codon] = codon_score_check.get(codon,0.0) + codon_shap_score_check
        bar_data_check = {}
        for codon in codon_list:
            if codon not in codon_score_check:
                bar_data_check[codon] = 0.0
            else:
                bar_data_check[codon] = codon_score_check[codon]
        phenotypic_distance_check1 = cdist(np.array(list(bar_data_check.values()), dtype=np.float32).reshape(1, -1), avi_medoid.reshape(1, -1), metric='cosine').flatten()
        ########################################################
        
        
        phenotypic_distance = batch_cosine_dist(row[codon_list].values.astype(np.float32).reshape(1, -1), avi_medoid)
        print (f'{seq_id} with Phenotypic Distance: {phenotypic_distance[0]}, {phenotypic_distance_check1[0]}')
        dists.append(phenotypic_distance_check1[0])

        # --- D. Build CSV Record ---
        record = {
            'Sequence_ID': seq_id,
            'Isolate_Name': row['Isolate_Name'],
            'Host': row['host'],
            'Mutation': 'E627K',
            'Phenotypic Distance': round(phenotypic_distance[0], 4),
            # 1. Specific Breakdown for 627
            'SHAP_Total_627': round(total_627_score, 4),
            'SHAP_627_Pos1_A': round(score_pos1, 4),
            'SHAP_627_Pos2_A': round(score_pos2, 4),
            'SHAP_627_Pos3_G': round(score_pos3, 4),
            # 2. Global Context
            'Top_20_Positive_Sites': top_20_str
        }
        supplementary_data.append(record)
        
        plot_dashboard_grid(
            scatter_data=codon_shap_by_pos,
            logo_array=seq_logo_data,
            bar_data=codon_bar_data,
            top_pos_data=top_20,
            codon=['Lysine K', 'AAA', 'AAG'],
            fig_title=f"{seq_id} - {row['Isolate_Name']} - H7N9",
            scatter_markers=[{'pos': 627, 'label': '627K', 'color': 'dodgerblue'}],
            logo_region=(1850, 1910),
            logo_highlight={'red': [(START_NUC_IDX, START_NUC_IDX + 3, "627K")]},
            subticks_frequency=5,
            is_at_rich_fn=is_at_rich,
            save_path=f"{results}/Mutation/H7N9_China/{seq_id}_H7N9_China.png"
        )

# ==========================================
# 3. SAVE TO CSV
# ==========================================
df_supp = pd.DataFrame(supplementary_data)
# Sort by the 3rd position (G) score, or Total 627 score? Usually Total is best.
df_supp = df_supp.sort_values(by='SHAP_Total_627', ascending=False)

csv_name = f"{results}/Supplementary_Table_E627K_H7N9.csv"
df_supp.to_csv(csv_name, index=False)
print(f"Mean Phenotypic Distance: {np.mean(dists)}")
print(f"Std Dev: {np.std(dists)}")
print(f"Done. Saved table to {csv_name}")
print(df_supp.head())


Starting analysis...
['GCA' 'GCC' 'GCG' 'GCT' 'TGC' 'TGT' 'GAC' 'GAT' 'GAA' 'GAG' 'TTC' 'TTT'
 'GGA' 'GGC' 'GGG' 'GGT' 'CAC' 'CAT' 'ATA' 'ATC' 'ATT' 'AAA' 'AAG' 'CTA'
 'CTC' 'CTG' 'CTT' 'TTA' 'TTG' 'ATG' 'AAC' 'AAT' 'CCA' 'CCC' 'CCG' 'CCT'
 'CAA' 'CAG' 'AGA' 'AGG' 'CGA' 'CGC' 'CGG' 'CGT' 'AGC' 'AGT' 'TCA' 'TCC'
 'TCG' 'TCT' 'ACA' 'ACC' 'ACG' 'ACT' 'GTA' 'GTC' 'GTG' 'GTT' 'TGG' 'TAC'
 'TAT']
Model prediction: 1
1..2280 0
EPI439488 with Phenotypic Distance: 1.063598871231079, 1.063598868572798
Figure saved to: /home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results/Mutation/H7N9_China/EPI439488_H7N9_China.png
Model prediction: 0
1..2280 0
EPI439495 with Phenotypic Distance: 1.209386944770813, 1.2093869228275627
Figure saved to: /home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results/Mutation/H7N9_China/EPI439495_H7N9_China.png
Model prediction: 1
1..2280 0
EPI439504 with Phenotypic Distance: 1.1353908777236938, 1.1353908173622658
Fi

In [24]:
dists

[1.063598868572798,
 1.2093869228275627,
 1.1353908173622658,
 0.8892423759858615,
 1.488165983568105,
 1.0178467064928385,
 1.3542728316872779,
 1.2050029663184967,
 1.2225935734099482,
 1.2207313535999238,
 1.115451514275328,
 1.1450315565852904,
 1.1391241739227085,
 1.1866654288560665,
 1.1397949428729433,
 1.457051735362971,
 1.4845781233662092,
 1.3460726046933265,
 1.435205821129696,
 1.0405084768459627,
 1.1349646368799313,
 1.2432489079662568,
 1.116401737902439,
 1.091001736298185,
 0.929153693662843]

In [25]:
np.mean(dists)

1.1924194996178095

In [26]:
np.std(dists)

0.1580061386370184

In [27]:
np.min(dists)

0.8892423759858615

In [28]:
np.max(dists)

1.488165983568105